# SPARQL queries against the atomRDF graph


Every sample, simulation cell and crystallographic property created by atomRDF is a real RDF triple, so anything you can express in SPARQL is available to you. This notebook shows three styles of querying:

1. Raw SPARQL (`kg.query("SELECT ...")`).
2. Term-builder for ontology-aware queries without writing SPARQL (`kg.query_sample`, `kg.query`).
3. Returning a sample object and operating on it.


In [ ]:
from atomrdf import KnowledgeGraph
import atomrdf.build as build


## Build a small heterogeneous database


In [ ]:
kg = KnowledgeGraph()
_ = build.bulk("Fe", cubic=True, graph=kg)
_ = build.bulk("Cu", cubic=True, graph=kg)
_ = build.bulk("Si", cubic=True, graph=kg)
_ = build.bulk("Mg", crystalstructure="hcp", graph=kg)
kg.n_samples


## 1. Raw SPARQL

What are the chemical species in the graph?


In [ ]:
q = """
PREFIX cmso: <http://purls.helmholtz-metadaten.de/cmso/>
SELECT DISTINCT ?symbol
WHERE {
    ?species cmso:hasElementSymbol ?symbol .
}
"""
kg.query(q)


Every sample with a cubic Bravais lattice and exactly two atoms in the unit cell:


In [ ]:
q = """
PREFIX cmso: <http://purls.helmholtz-metadaten.de/cmso/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
SELECT ?sample ?symbol
WHERE {
    ?sample  cmso:hasNumberOfAtoms ?n ;
             cmso:hasMaterial      ?m .
    ?m       cmso:hasStructure     ?s .
    ?s       cmso:hasSpaceGroupSymbol ?symbol .
    FILTER (?n = "2"^^xsd:integer)
}
"""
kg.query(q)


## 2. Term builder (when the ontology network is available)

`kg.terms.cmso.AtomicScaleSample` lets you express the same query without typing SPARQL. It requires the ontology network to be reachable at construction time — if it is not (e.g. behind a strict firewall), `kg.terms` will be `None` and you should fall back to the raw SPARQL form above.

```python
kg.query(
    kg.terms.cmso.AtomicScaleSample,
    [
        kg.terms.cmso.hasSpaceGroupSymbol,
        kg.terms.cmso.hasNumberOfAtoms == 2,
    ],
)
```


## 3. Return a single sample and write it out


In [ ]:
q = """
PREFIX cmso: <http://purls.helmholtz-metadaten.de/cmso/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
SELECT ?sample
WHERE {
    ?sample cmso:hasNumberOfAtoms ?n .
    FILTER (?n = "2"^^xsd:integer)
}
"""
df = kg.query(q)
df


In [ ]:
sample = df['sample'].values[0]
kg.to_file(sample, "selected.poscar", format="vasp")
! head -10 selected.poscar
